# PROTEUS: classifier-only ablation of the submitted Lag4 forest

Private execution. Same fixed geometry,50 trees,subject bootstrap,8 LOSO folds,one later session and original train-only public test. MLP32 runs on CUDA; five literature-grounded alternatives run on CPU. No test tuning. The source bundle and all inputs are hash-verified.

In [ ]:
import os, sys, json, base64, hashlib, zipfile, io, pathlib, subprocess, importlib.metadata
os.environ.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",CUBLAS_WORKSPACE_CONFIG=":4096:8")
import torch
assert torch.cuda.is_available(), "Enable a CUDA GPU for the requested MLP comparison"
print("GPU", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
project=pathlib.Path("/kaggle/working/riemann-project")
project.mkdir(parents=True,exist_ok=True)
sources=list(pathlib.Path("/kaggle/input").rglob("source-integrity.json"))
assert len(sources)==1, sources
source_manifest=sources[0].read_bytes()
assert hashlib.sha256(source_manifest).hexdigest()=="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81"
for rel,digest in json.loads(source_manifest).items():
    path=pathlib.PurePosixPath(rel)
    assert not path.is_absolute() and ".." not in path.parts
    data=(sources[0].parent/rel).read_bytes()
    assert hashlib.sha256(data).hexdigest()==digest,rel
    dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
(project/"source-integrity.json").write_bytes(source_manifest)
matches=list(pathlib.Path("/kaggle/input").rglob("input-integrity.json"))
assert len(matches)==1, matches
inputs=matches[0].parent
manifest=json.loads(matches[0].read_text())
for name,digest in manifest["files_sha256"].items():
    h=hashlib.sha256()
    with (inputs/name).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    assert h.hexdigest()==digest,name
repro=project/"reproduction"
(repro/"data").mkdir(exist_ok=True)
for split in ["train","val","test"]:
    link=repro/"data"/f"proteus_{split}.npz"
    if not link.exists():link.symlink_to(inputs/link.name)
cache=repro/"temporal_spd_research/cache"
if not cache.exists():cache.symlink_to(inputs,target_is_directory=True)
envdir=pathlib.Path("/tmp/classifier-env")
subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(envdir)],check=True)
python=str(envdir/"bin/python")
pins=["numpy==2.5.4","scipy==1.18.1","scikit-learn==1.9.1","pyriemann==0.12","pandas==3.1.0","threadpoolctl==3.7.0"]
subprocess.run([sys.executable,"-m","pip","--python",python,"install","--disable-pip-version-check","--quiet",*pins],check=True)
subprocess.run([python,"-c","import sys,torch,importlib.metadata as m;print('Numerical runtime',sys.version,{k:m.version(k) for k in ['numpy','scipy','scikit-learn','pyriemann','pandas','torch']});assert torch.cuda.is_available()"],check=True)
out=pathlib.Path("/kaggle/working/classifier-results");out.mkdir(exist_ok=True)
env=dict(os.environ,CLASSIFIER_OUTPUT=str(out),CLASSIFIER_CACHE="/tmp/classifier-features",PYTHONUNBUFFERED="1",MPLCONFIGDIR="/tmp/matplotlib-classifier")
runtime=dict(gpu=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],torch=torch.__version__,python=sys.version,source_manifest_sha256=hashlib.sha256(source_manifest).hexdigest(),input_manifest=manifest)
(out/"runtime.json").write_text(json.dumps(runtime,indent=2))
print("Verified all source/input hashes. Original data and submitted model are attached.",flush=True)


## Verify the implementation before accuracy experiments

In [ ]:
experiment=repro/"classifier_ablation"
test_files=["test_classifiers.py","test_resume.py","test_features.py","test_gpu_mlp.py","test_metrics.py"]
subprocess.run([python,"-m","pytest","-q",*test_files,"--tb=short"],cwd=experiment,env=env,check=True)
print("Integrity tests passed; starting fixed classifier comparison",flush=True)


## Run CV,freeze its selection,then evaluate public test

In [ ]:
subprocess.run([python,"-u",str(experiment/"run.py"),"--phase","all","--workers","3"],cwd=experiment,env=env,check=True)


## Independently verify metrics and summarize paired differences

In [ ]:
subprocess.run([python,"-u",str(experiment/"analyze.py")],cwd=experiment,env=env,check=True)
print((out/"REPORT.md").read_text())
print("Outputs:",sorted(p.name for p in out.iterdir()))
